In [8]:
import sys
print(sys.executable)

/anaconda/envs/azureml_py310_sdkv2/bin/python


In [9]:
!{sys.executable} -m pip install azure-cognitiveservices-vision-customvision --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip


In [ ]:
from azure.cognitiveservices.vision.customvision.training import CustomVisionTrainingClient
from msrest.authentication import ApiKeyCredentials

TRAINING_ENDPOINT = "your_training_endpoint"
TRAINING_KEY = "your_training_key"
PROJECT_ID = "your_project_id"

credentials = ApiKeyCredentials(in_headers={"Training-key": TRAINING_KEY})
trainer = CustomVisionTrainingClient(TRAINING_ENDPOINT, credentials)

project = trainer.get_project(PROJECT_ID)
print(f"Csatlakozva a projekthez: {project.name}")

Csatlakozva a projekthez: skin-lesion-classifier


In [2]:
import pandas as pd

metadata = pd.read_csv("./ham10000_data/HAM10000_metadata.csv")

sampled = metadata.groupby("dx", group_keys=False).apply(
    lambda x: x.sample(n=min(100, len(x)), random_state=42)
)

print(sampled["dx"].value_counts())
print(f"\nÖsszes kiválasztott kép: {len(sampled)}")

akiec    100
bcc      100
bkl      100
df       100
mel      100
nv       100
vasc     100
Name: dx, dtype: int64

Összes kiválasztott kép: 700


In [3]:
tags = {}
for category in sampled["dx"].unique():
    tag = trainer.create_tag(PROJECT_ID, category)
    tags[category] = tag
    print(f"Tag létrehozva: {category} (id: {tag.id})")

Tag létrehozva: akiec (id: 64b91e4a-faf1-42cd-908b-eb6512f2f865)
Tag létrehozva: bcc (id: 35e3b0b8-3cdc-4ef4-951e-bbd823b21c08)
Tag létrehozva: bkl (id: 1ce0cddc-74ba-4938-8dd1-ea9943da61d8)
Tag létrehozva: df (id: 1f8d03ff-5aa7-42e0-abcf-c03b7acb2349)
Tag létrehozva: mel (id: afe4cb28-3a80-4d54-bf8d-7a80fb8f2a45)
Tag létrehozva: nv (id: 7bbdef29-806c-4cfd-9a87-ce1554ad68a7)
Tag létrehozva: vasc (id: 31a24a77-77e9-47e0-95e2-e22cf26fe73d)


In [ ]:
import os
from azure.cognitiveservices.vision.customvision.training.models import ImageFileCreateEntry, ImageFileCreateBatch

def find_image_path(image_id):
    for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        path = f"./ham10000_data/{folder}/{image_id}.jpg"
        if os.path.exists(path):
            return path
    return None

image_entries = []
skipped = 0

for _, row in sampled.iterrows():
    img_path = find_image_path(row["image_id"])
    if img_path is None:
        skipped += 1
        continue
    with open(img_path, "rb") as f:
        image_entries.append(
            ImageFileCreateEntry(
                name=row["image_id"],
                contents=f.read(),
                tag_ids=[tags[row["dx"]].id]
            )
        )

print(f"Feltöltésre kész képek: {len(image_entries)}, kihagyva: {skipped}")

batch_size = 64
for i in range(0, len(image_entries), batch_size):
    batch = image_entries[i:i+batch_size]
    upload_result = trainer.create_images_from_files(PROJECT_ID, ImageFileCreateBatch(images=batch))
    if not upload_result.is_batch_successful:
        for img in upload_result.images:
            if img.status != "OK":
                print(f"Hiba: {img.source_url}, státusz: {img.status}")
    print(f"Batch {i//batch_size + 1} feltöltve ({i+len(batch)}/{len(image_entries)})")

print("Feltöltés kész!")

Feltöltésre kész képek: 700, kihagyva: 0
Batch 1 feltöltve (64/700)
Batch 2 feltöltve (128/700)
Batch 3 feltöltve (192/700)
Batch 4 feltöltve (256/700)
Batch 5 feltöltve (320/700)
Batch 6 feltöltve (384/700)
Batch 7 feltöltve (448/700)
Batch 8 feltöltve (512/700)
Batch 9 feltöltve (576/700)
Batch 10 feltöltve (640/700)


In [ ]:
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from msrest.authentication import ApiKeyCredentials

PREDICTION_ENDPOINT = "your_pred_endpoint"
PREDICTION_KEY = "your_pred_key"
PUBLISHED_NAME = "your_published_name" 

prediction_credentials = ApiKeyCredentials(in_headers={"Prediction-key": PREDICTION_KEY})
predictor = CustomVisionPredictionClient(PREDICTION_ENDPOINT, prediction_credentials)

test_image_path = "./ham10000_data/HAM10000_images_part_1/" + sampled.iloc[0]["image_id"] + ".jpg"

with open(test_image_path, "rb") as image_data:
    results = predictor.classify_image(PROJECT_ID, PUBLISHED_NAME, image_data.read())

print(f"Valós kategória: {sampled.iloc[0]['dx']}")
print("\nPredikciók:")
for prediction in results.predictions:
    print(f"{prediction.tag_name}: {prediction.probability:.2%}")

Valós kategória: akiec

Predikciók:
bkl: 97.62%
nv: 1.05%
mel: 0.89%
akiec: 0.37%
df: 0.04%
bcc: 0.02%
vasc: 0.01%
